# Fuel Delivery Optimization with Amazon SageMaker & AutoGluon

This notebook demonstrates an end-to-end fuel delivery optimization pipeline that replaces 
the simple heuristic (`avg_consumption × 1.20`) with ML-driven delivery recommendations using:

- **AutoGluon-TimeSeries** — Ensemble forecasting (statistical + ML + deep learning)
- **Amazon SageMaker** — Managed training and inference at scale
- **Delivery Optimization** — Converts forecasts into optimal delivery schedules

## Contents

| # | Section | Environment |
|---|---------|-------------|
| 1 | Setup & Configuration | Local |
| 2 | Data Generation | Local |
| 3 | Validation & EDA | Local |
| 4 | Feature Engineering | Local |
| 5 | Local Training | Local |
| 6 | SageMaker Training | AWS |
| 7 | Deployment | AWS |
| 8 | Inference | AWS / Local |
| 9 | Delivery Optimization | Local |
| 10 | Evaluation | Local |
| 11 | Visualization | Local |
| 12 | Cleanup | AWS |

---
## 1. Setup & Configuration

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -q autogluon.timeseries sagemaker boto3 pandas matplotlib holidays pyyaml

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

from data_preparation import FuelDeliveryDataProcessor, prepare_forecast_data
from utils import (
    generate_fuel_delivery_data,
    evaluate_forecasts,
    load_config,
    calculate_business_metrics,
    plot_consumption_forecast,
    plot_delivery_schedule,
    plot_tank_inventory_projection,
    plot_heuristic_vs_ml_comparison,
)
from optimization import DeliveryConstraints, DeliveryOptimizer, DeliveryScheduler
from backtesting import BacktestEngine
from calendar_us import get_us_holidays, add_holiday_features

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")

print("All modules imported successfully.")

In [ ]:
# Configuration — matches config/config.yaml
CONFIG = {
    # Data
    "freq": "D",
    "prediction_length": 10,
    # Training
    "presets": "medium_quality",
    "time_limit": 600,  # seconds
    # Evaluation
    "quantiles": [0.1, 0.25, 0.5, 0.75, 0.9],
    # AWS (update for SageMaker sections)
    "region": "us-east-1",
    "instance_type": "ml.m5.xlarge",
    "s3_prefix": "fuel-delivery-optimization",
}

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

print("Configuration loaded.")

---
## 2. Data Generation

Generate synthetic fuel delivery data for 50 tanks over 365 days. Three tank types:

| Type | Consumption (gal/day) | Capacity (gal) |
|------|----------------------|-----------------|
| Residential | 2 – 8 | 275 |
| Commercial | 8 – 25 | 1,000 |
| Industrial | 25 – 60 | 5,000 |

In [ ]:
df = generate_fuel_delivery_data(
    n_tanks=50,
    n_days=365,
    start_date="2024-01-01",
    seed=42,
)

print(f"Shape: {df.shape}")
print(f"Tanks: {df['tank_id'].nunique()}")
print(f"Date range: {df['reading_date'].min()} to {df['reading_date'].max()}")
df.head()

In [ ]:
# Consumption statistics by tank type
print("Daily Consumption by Tank Type (gal/day):")
print("=" * 55)
print(
    df.groupby("tank_type")["daily_consumption"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(2)
)

print(f"\nTank capacities: {sorted(df['tank_capacity'].unique())}")
print(f"Delivery events: {(df.get('delivery_volume', pd.Series([0])) > 0).sum():,}")
print(f"Holiday observations: {df['is_holiday'].sum():,} ({df['is_holiday'].mean():.1%})")

In [ ]:
# Visualize consumption for one tank of each type
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, tank_type in zip(axes, ["residential", "commercial", "industrial"]):
    subset = df[df["tank_type"] == tank_type]
    sample_tank = subset["tank_id"].unique()[0]
    tank_df = subset[subset["tank_id"] == sample_tank]

    ax.plot(
        pd.to_datetime(tank_df["reading_date"]),
        tank_df["daily_consumption"],
        linewidth=0.8,
    )
    ax.set_title(f"{sample_tank} ({tank_type})")
    ax.set_ylabel("Consumption (gal)")

axes[-1].set_xlabel("Date")
fig.suptitle("Daily Fuel Consumption by Tank Type", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

---
## 3. Validation & Exploratory Data Analysis

In [ ]:
# Initialize data processor
processor = FuelDeliveryDataProcessor(
    item_id_column="tank_id",
    timestamp_column="reading_date",
    target_column="daily_consumption",
    freq="D",
)

# Validate time series
validation = processor.validate_time_series(df)
print("Validation Results:")
for key, value in validation.items():
    print(f"  {key}: {value}")

In [ ]:
# Consumption distributions by tank type
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, tank_type in zip(axes, ["residential", "commercial", "industrial"]):
    subset = df[df["tank_type"] == tank_type]["daily_consumption"]
    ax.hist(subset, bins=40, edgecolor="white", alpha=0.8)
    ax.axvline(subset.mean(), color="red", linestyle="--", label=f"Mean: {subset.mean():.1f}")
    ax.set_title(f"{tank_type.title()}")
    ax.set_xlabel("Daily Consumption (gal)")
    ax.legend()

fig.suptitle("Consumption Distributions by Tank Type", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Weekly and seasonal patterns
df["_date"] = pd.to_datetime(df["reading_date"])
df["_dow"] = df["_date"].dt.day_name()
df["_month"] = df["_date"].dt.month

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Day-of-week
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_avg = df.groupby("_dow")["daily_consumption"].mean().reindex(dow_order)
ax1.bar(range(7), dow_avg.values, color="steelblue")
ax1.set_xticks(range(7))
ax1.set_xticklabels([d[:3] for d in dow_order])
ax1.set_title("Average Consumption by Day of Week")
ax1.set_ylabel("Consumption (gal)")

# Monthly seasonality
month_avg = df.groupby(["_month", "tank_type"])["daily_consumption"].mean().unstack()
month_avg.plot(ax=ax2, marker="o")
ax2.set_title("Monthly Consumption by Tank Type")
ax2.set_xlabel("Month")
ax2.set_ylabel("Consumption (gal)")
ax2.set_xticks(range(1, 13))

fig.tight_layout()
plt.show()

# Clean up temp columns
df.drop(columns=["_date", "_dow", "_month"], inplace=True)

In [ ]:
# Holiday vs non-holiday consumption
if "is_holiday" in df.columns:
    holiday_stats = (
        df.groupby(["tank_type", "is_holiday"])["daily_consumption"]
        .mean()
        .unstack()
        .rename(columns={0: "Non-Holiday", 1: "Holiday"})
    )
    holiday_stats["Diff %"] = (
        (holiday_stats["Holiday"] - holiday_stats["Non-Holiday"])
        / holiday_stats["Non-Holiday"]
        * 100
    )
    print("Average Consumption: Holiday vs Non-Holiday")
    print("=" * 50)
    print(holiday_stats.round(2))

---
## 4. Feature Engineering

In [ ]:
# Run the full feature pipeline
df_features = processor.build_feature_pipeline(df)

print(f"Original columns: {len(df.columns)}")
print(f"After feature engineering: {len(df_features.columns)}")
print(f"\nNew features:")
new_cols = sorted(set(df_features.columns) - set(df.columns))
for col in new_cols:
    print(f"  - {col}")

In [ ]:
# Feature statistics for a sample tank
sample_tank = df_features["tank_id"].unique()[0]
tank_feats = df_features[df_features["tank_id"] == sample_tank]

feature_cols = [
    col for col in new_cols
    if col in tank_feats.columns and tank_feats[col].dtype in ["float64", "int64"]
]

print(f"Feature summary for {sample_tank}:")
print("=" * 60)
print(tank_feats[feature_cols].describe().round(3).T[["mean", "std", "min", "max"]])

In [ ]:
# Correlation of features with target
numeric_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()
if "daily_consumption" in numeric_cols:
    corr = df_features[numeric_cols].corr()["daily_consumption"].drop("daily_consumption").sort_values()
    top_features = pd.concat([corr.head(5), corr.tail(5)])

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ["steelblue" if v > 0 else "salmon" for v in top_features.values]
    top_features.plot.barh(ax=ax, color=colors)
    ax.set_title("Top Feature Correlations with Daily Consumption")
    ax.set_xlabel("Correlation")
    ax.axvline(0, color="black", linewidth=0.5)
    fig.tight_layout()
    plt.show()

In [ ]:
# Train / test split
train_df, test_df = processor.train_test_split(
    df,
    prediction_length=CONFIG["prediction_length"],
)

print(f"Training data: {len(train_df):,} rows ({train_df['tank_id'].nunique()} tanks)")
print(f"Test data:     {len(test_df):,} rows ({test_df['tank_id'].nunique()} tanks)")

# Save to disk
train_df.to_csv(DATA_DIR / "train.csv", index=False)
test_df.to_csv(DATA_DIR / "test.csv", index=False)
print(f"\nSaved to {DATA_DIR}/train.csv and {DATA_DIR}/test.csv")

---
## 5. Local Training

Train an AutoGluon-TimeSeries ensemble locally for rapid iteration.

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

train_df["reading_date"] = pd.to_datetime(train_df["reading_date"])

train_ts = TimeSeriesDataFrame.from_data_frame(
    train_df,
    id_column="tank_id",
    timestamp_column="reading_date",
)

print(f"TimeSeriesDataFrame: {train_ts.num_items} tanks, {len(train_ts):,} rows")

In [ ]:
predictor = TimeSeriesPredictor(
    path="../models/local_model",
    prediction_length=CONFIG["prediction_length"],
    freq=CONFIG["freq"],
    target="daily_consumption",
    eval_metric="MASE",
    quantile_levels=CONFIG["quantiles"],
)

predictor.fit(
    train_data=train_ts,
    presets="fast_training",  # Use 'medium_quality' or higher for production
    time_limit=300,
)

In [ ]:
leaderboard = predictor.leaderboard(train_ts)
print("Model Leaderboard:")
display(leaderboard)

In [ ]:
# Generate 10-day consumption forecasts
predictions = predictor.predict(train_ts)
predictions = predictions.reset_index()

print(f"Predictions: {len(predictions):,} rows, {predictions['item_id'].nunique()} tanks")
print(f"Columns: {list(predictions.columns)}")
predictions.head()

---
## 6. SageMaker Training (Production)

> **Note**: This section requires AWS credentials and a SageMaker execution role.

In [ ]:
import os
from sagemaker_pipeline import FuelDeliveryPipeline

src_path = os.path.abspath("../src")
print(f"Source dir: {src_path}")

pipeline = FuelDeliveryPipeline(
    region=CONFIG["region"],
    prefix=CONFIG["s3_prefix"],
    source_dir=src_path,
)
pipeline.source_dir = "../src"

In [ ]:
training_job = pipeline.train(
    train_data="../data/train.csv",
    hyperparameters={
        "prediction_length": CONFIG["prediction_length"],
        "freq": CONFIG["freq"],
        "presets": CONFIG["presets"],
        "time_limit": CONFIG["time_limit"],
        "target_column": "daily_consumption",
        "item_id_column": "tank_id",
        "timestamp_column": "reading_date",
    },
    instance_type=CONFIG["instance_type"],
    wait=True,
)

print(f"Training completed: {training_job}")
print(f"Model artifacts: {pipeline.model_data}")

---
## 7. Deployment

In [ ]:
predictor_endpoint = pipeline.deploy(
    instance_type="ml.m5.large",
    instance_count=1,
    wait=True,
)

print(f"Endpoint deployed: {pipeline.endpoint_name}")

In [ ]:
# Quick smoke test — predict for a single tank
sample_tank = train_df["tank_id"].unique()[0]
sample_input = train_df[train_df["tank_id"] == sample_tank][
    ["tank_id", "reading_date", "daily_consumption"]
].copy()

endpoint_preds = predictor_endpoint.predict(sample_input)
print(f"Endpoint predictions for {sample_tank}:")
display(endpoint_preds.head())

---
## 8. Inference

Generate consumption forecasts for all tanks (using local model for the demo).

In [ ]:
# Use local predictions from Section 5
# For endpoint predictions, replace with: predictor_endpoint.predict(data)

all_predictions = predictions.copy()

print(f"Total predictions: {len(all_predictions):,}")
print(f"Unique tanks: {all_predictions['item_id'].nunique()}")
print(f"Forecast horizon: {all_predictions.groupby('item_id')['timestamp'].count().iloc[0]} days")

In [ ]:
# Aggregate daily consumption forecast across all tanks
daily_forecast = all_predictions.groupby("timestamp").agg(
    {"mean": "sum", "0.1": "sum", "0.9": "sum"}
).reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_forecast["timestamp"], daily_forecast["mean"], label="Forecast (mean)")
ax.fill_between(
    daily_forecast["timestamp"],
    daily_forecast["0.1"],
    daily_forecast["0.9"],
    alpha=0.25,
    label="80% Prediction Interval",
)
ax.set_title("Total Fuel Consumption Forecast (All Tanks)")
ax.set_xlabel("Date")
ax.set_ylabel("Total Consumption (gal)")
ax.legend()
plt.show()

In [ ]:
# Preview per-tank forecasts (first 5 tanks)
for tank_id in all_predictions["item_id"].unique()[:5]:
    tp = all_predictions[all_predictions["item_id"] == tank_id]
    print(f"{tank_id}: mean={tp['mean'].mean():.2f}, p10={tp['0.1'].mean():.2f}, p90={tp['0.9'].mean():.2f}")

---
## 9. Delivery Optimization

Convert consumption forecasts into delivery recommendations. The optimizer uses the **p90** 
(90th-percentile) forecast for conservative volume estimates and classifies urgency based on 
the current tank level.

In [ ]:
# Initialize optimizer with default constraints
constraints = DeliveryConstraints()
optimizer = DeliveryOptimizer(constraints)

print("Delivery Constraints:")
print(f"  Min delivery volume: {constraints.min_delivery_volume} gal")
print(f"  Max delivery volume: {constraints.max_delivery_volume} gal")
print(f"  Target fill %: {constraints.target_fill_pct:.0%}")
print(f"  Reorder threshold: {constraints.reorder_pct:.0%}")
print(f"  Emergency threshold: {constraints.emergency_pct:.0%}")
print(f"  Safety buffer: {constraints.safety_buffer_days} days")

In [ ]:
# Single-tank example
sample_tank = train_df["tank_id"].unique()[0]
tank_data = train_df[train_df["tank_id"] == sample_tank]
tank_predictions = all_predictions[all_predictions["item_id"] == sample_tank]

current_level = tank_data["current_level"].iloc[-1]
tank_capacity = tank_data["tank_capacity"].iloc[-1]

print(f"Tank: {sample_tank}")
print(f"  Type: {tank_data['tank_type'].iloc[0]}")
print(f"  Capacity: {tank_capacity:.0f} gal")
print(f"  Current level: {current_level:.0f} gal ({current_level / tank_capacity:.0%})")

rec = optimizer.calculate_optimal_delivery(
    tank_id=sample_tank,
    current_level=current_level,
    tank_capacity=tank_capacity,
    forecast=tank_predictions,
)

print(f"\nML Recommendation:")
print(f"  Urgency:  {rec.urgency}")
print(f"  Volume:   {rec.volume:.0f} gal")
print(f"  Fill %:   {rec.fill_pct:.0%}")
print(f"  Coverage: {rec.coverage_days:.0f} days")
print(f"  Reason:   {rec.reasoning}")

In [ ]:
# Compare ML vs heuristic for the same tank
hist_avg = tank_data["daily_consumption"].mean()

comparison = optimizer.compare_with_heuristic(
    tank_id=sample_tank,
    current_level=current_level,
    tank_capacity=tank_capacity,
    forecast=tank_predictions,
    historical_avg_consumption=hist_avg,
)

print(f"ML vs Heuristic (avg × 1.20) for {sample_tank}:")
print(f"  ML Volume:        {comparison['ml_volume']:.0f} gal")
print(f"  Heuristic Volume: {comparison['heuristic_volume']:.0f} gal")
print(f"  Volume Savings:   {comparison['volume_savings']:.0f} gal ({comparison['volume_savings_pct']:.1f}%)")

In [ ]:
# Multi-tank delivery schedule
scheduler = DeliveryScheduler(constraints)

# Build tank info for all tanks
tank_info = []
for tank_id in train_df["tank_id"].unique():
    td = train_df[train_df["tank_id"] == tank_id]
    tp = all_predictions[all_predictions["item_id"] == tank_id]

    if len(tp) == 0:
        continue

    tank_info.append({
        "tank_id": tank_id,
        "current_level": td["current_level"].iloc[-1],
        "tank_capacity": td["tank_capacity"].iloc[-1],
        "forecast": tp,
    })

schedule = scheduler.create_schedule(tank_info)

print(f"Delivery Schedule: {len(schedule)} tanks")
print(f"\nUrgency breakdown:")
urgency_counts = pd.Series([r.urgency for r in schedule]).value_counts()
for urgency, count in urgency_counts.items():
    print(f"  {urgency}: {count}")

total_volume = sum(r.volume for r in schedule)
print(f"\nTotal scheduled volume: {total_volume:,.0f} gal")

In [ ]:
# Show top-priority deliveries
schedule_df = pd.DataFrame([
    {
        "tank_id": r.tank_id,
        "urgency": r.urgency,
        "volume_gal": round(r.volume),
        "fill_pct": f"{r.fill_pct:.0%}",
        "coverage_days": round(r.coverage_days),
        "reasoning": r.reasoning,
    }
    for r in schedule[:15]
])

print("Top 15 Priority Deliveries:")
display(schedule_df)

---
## 10. Evaluation

In [ ]:
# Forecast accuracy metrics
if len(test_df) > 0:
    test_df["reading_date"] = pd.to_datetime(test_df["reading_date"])

    metrics = evaluate_forecasts(
        test_df,
        all_predictions,
        target_column="daily_consumption",
        item_id_column="tank_id",
        timestamp_column="reading_date",
    )

    print("Forecast Evaluation Metrics:")
    print("=" * 40)
    for metric, value in metrics.items():
        print(f"  {metric:20s}: {value:.4f}")
else:
    print("No test data available for evaluation.")

In [ ]:
# Business metrics
biz_metrics = calculate_business_metrics(
    schedule,
    tank_info,
)

print("Business Metrics:")
print("=" * 40)
for metric, value in biz_metrics.items():
    if isinstance(value, float):
        print(f"  {metric:30s}: {value:.2f}")
    else:
        print(f"  {metric:30s}: {value}")

In [ ]:
# Walk-forward backtest: ML vs Heuristic
backtest_engine = BacktestEngine(
    constraints=constraints,
    prediction_length=CONFIG["prediction_length"],
)

# Run backtest on a sample tank
sample_tank = train_df["tank_id"].unique()[0]
tank_history = df[df["tank_id"] == sample_tank].copy()

backtest_results = backtest_engine.run(
    tank_id=sample_tank,
    history=tank_history,
    tank_capacity=tank_history["tank_capacity"].iloc[0],
)

print(f"Backtest Results for {sample_tank}:")
print("=" * 50)
for key, value in backtest_results.metrics.items():
    if isinstance(value, float):
        print(f"  {key:30s}: {value:.3f}")
    else:
        print(f"  {key:30s}: {value}")

In [ ]:
# Aggregate ML vs Heuristic comparison across all tanks
ml_volumes = []
heuristic_volumes = []

for info in tank_info:
    tid = info["tank_id"]
    td = train_df[train_df["tank_id"] == tid]
    hist_avg = td["daily_consumption"].mean()

    comp = optimizer.compare_with_heuristic(
        tank_id=tid,
        current_level=info["current_level"],
        tank_capacity=info["tank_capacity"],
        forecast=info["forecast"],
        historical_avg_consumption=hist_avg,
    )
    ml_volumes.append(comp["ml_volume"])
    heuristic_volumes.append(comp["heuristic_volume"])

total_ml = sum(ml_volumes)
total_heur = sum(heuristic_volumes)
savings = total_heur - total_ml

print("Fleet-wide ML vs Heuristic Summary:")
print("=" * 45)
print(f"  Total ML volume:        {total_ml:>10,.0f} gal")
print(f"  Total Heuristic volume: {total_heur:>10,.0f} gal")
print(f"  Volume savings:         {savings:>10,.0f} gal ({savings / total_heur * 100:.1f}%)")

---
## 11. Visualization

In [ ]:
# Consumption forecast with actuals
sample_tank = train_df["tank_id"].unique()[0]

fig = plot_consumption_forecast(
    historical=train_df,
    predictions=all_predictions,
    tank_id=sample_tank,
    actuals=test_df if len(test_df) > 0 else None,
    title=f"Consumption Forecast — {sample_tank}",
)
plt.show()

In [ ]:
# Forecasts for multiple tanks (one per type)
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for ax, tank_type in zip(axes, ["residential", "commercial", "industrial"]):
    type_tanks = train_df[train_df["tank_type"] == tank_type]["tank_id"].unique()
    if len(type_tanks) == 0:
        continue
    tid = type_tanks[0]

    hist = train_df[train_df["tank_id"] == tid]
    pred = all_predictions[all_predictions["item_id"] == tid]

    ax.plot(pd.to_datetime(hist["reading_date"]), hist["daily_consumption"],
            label="Historical", color="steelblue", linewidth=0.8)
    ax.plot(pred["timestamp"], pred["mean"], label="Forecast", color="darkorange")
    ax.fill_between(pred["timestamp"], pred["0.1"], pred["0.9"],
                    alpha=0.2, color="darkorange", label="80% PI")

    if len(test_df) > 0:
        act = test_df[test_df["tank_id"] == tid]
        ax.scatter(pd.to_datetime(act["reading_date"]), act["daily_consumption"],
                   color="green", s=20, label="Actual", zorder=5)

    ax.set_title(f"{tid} ({tank_type})")
    ax.set_ylabel("Consumption (gal)")
    ax.legend(loc="upper left", fontsize=8)

axes[-1].set_xlabel("Date")
fig.suptitle("10-Day Consumption Forecasts by Tank Type", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Inventory projection for a sample tank
sample_tank = train_df["tank_id"].unique()[0]
tank_data = train_df[train_df["tank_id"] == sample_tank]
tank_preds = all_predictions[all_predictions["item_id"] == sample_tank]

fig = plot_tank_inventory_projection(
    tank_id=sample_tank,
    current_level=tank_data["current_level"].iloc[-1],
    tank_capacity=tank_data["tank_capacity"].iloc[0],
    forecast=tank_preds,
    reorder_pct=constraints.reorder_pct,
    emergency_pct=constraints.emergency_pct,
)
plt.show()

In [ ]:
# Delivery schedule overview
fig = plot_delivery_schedule(schedule[:20])
plt.show()

In [ ]:
# ML vs Heuristic volume comparison
fig = plot_heuristic_vs_ml_comparison(
    ml_volumes=ml_volumes,
    heuristic_volumes=heuristic_volumes,
    tank_ids=[info["tank_id"] for info in tank_info],
)
plt.show()

In [ ]:
# Urgency distribution
urgency_counts = pd.Series([r.urgency for r in schedule]).value_counts()

colors = {
    "emergency": "#d62728",
    "urgent": "#ff7f0e",
    "normal": "#2ca02c",
    "low": "#1f77b4",
}

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    urgency_counts.values,
    labels=urgency_counts.index,
    colors=[colors.get(u, "gray") for u in urgency_counts.index],
    autopct="%1.0f%%",
    startangle=90,
)
ax.set_title("Delivery Urgency Distribution")
plt.show()

---
## 12. Cleanup

Delete SageMaker resources when done to avoid charges.

In [ ]:
# Uncomment to delete SageMaker resources

# pipeline.cleanup(
#     delete_endpoint=True,
#     delete_model=True,
# )
# print("SageMaker resources cleaned up.")

---
## Next Steps

1. **Use your own data** — Replace `generate_fuel_delivery_data()` with your actual tank readings
2. **Tune models** — Increase `time_limit` and use `high_quality` or `best_quality` presets
3. **Configure holidays** — Set `state` in `config/config.yaml` for state-specific holidays
4. **Adjust constraints** — Tune safety buffers, reorder thresholds, and fill targets in config
5. **Scale with SageMaker** — Use sections 6–7 for production training and real-time inference
6. **Automate** — Set up SageMaker Pipelines for scheduled retraining and batch inference